## Regime Detection

Description:     
We detect market regimes (up, down, sideways) using a simple rule—if a short moving average is above a longer one, label = +1 (“up” regime), if below, label = -1 (“down” regime), otherwise 0 (“sideways”). By identifying these regimes, we can adapt trading strategies to different market states or selectively trade only in certain conditions.

#### 📌 Important Note:
This notebook contains *interactive charts generated using Vectorbt.  
GitHub does not display interactive Plotly charts, so the graphs will not be visible here.  

✅ To view the charts, please download this notebook and run it on your local machine.  
Make sure you have Vectorbt and its dependencies installed to regenerate the visualizations.


## Part 1: Data & Feature Engineering

**Objective:**  
Load raw price data (MetaTrader 5 or CSV) and transform it into a feature-rich dataset.

**Tasks:**
- Fetch historical bars  
- Apply `ta.add_all_ta_features` or custom features  
- (Optionally) create specific labels (multi-bar, double-barrier, regime, etc.)  
- Clean/prepare the final feature matrix **X** and target **y**  

In [ ]:
# REGIME DETECTION + LabelEncoder approach for SHIFTED labels
import sys
import os
import warnings
from pathlib import Path

# ---------------------------------------------------------------------------
# 1) SET PROJECT ROOT AND UPDATE PATH/WORKING DIRECTORY
# ---------------------------------------------------------------------------
project_root = Path.cwd().parent.parent  # Adjust if your notebook is in notebooks/time_series
sys.path.append(str(project_root))
os.chdir(str(project_root))
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import MetaTrader5 as mt5
import vectorbt as vbt
import joblib

# Our modules
from data.data_loader import get_data_mt5
from features.feature_engineering import add_all_ta_features
from models.model_training import (
    select_features_rf_reg,
    walk_forward_splits
)
from backtests.simple_backtest import simulate_trading, calculate_sharpe_ratio
from features.labeling_schemes import create_labels_regime_detection  # or define inline




# Sklearn / Models
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB


###########################################################
# 1) DATA LOADING & FEATURE ENGINEERING
###########################################################
if not mt5.initialize():
    print("Failed to initialize MT5")
else:
    data = get_data_mt5(symbol="BTCUSD", timeframe=mt5.TIMEFRAME_H4, n_bars=2000, start_pos=2000)
    mt5.shutdown()

df = add_all_ta_features(data)

###########################################################
# 2) Regime Detection Labeling
###########################################################


df_lbl = create_labels_regime_detection(df, short_window=20, long_window=50)

# X => all features except the label columns
X = df_lbl.drop(columns=["regime_label", "ma_short", "ma_long"])
y = df_lbl["regime_label"]  # in {-1,0,+1}

###########################################################
# SHIFT LABELS? 
# We'll SHIFT them to [0,1,2] in each fold with LabelEncoder
# so we can handle folds with only two classes, e.g. {0,2}.
###########################################################

###########################################################
# 3) WALK-FORWARD SPLITS
###########################################################
folds = walk_forward_splits(X, y, n_splits=3)
print(f"Number of folds created: {len(folds)}")

###########################################################
# 4) DEFINE CLASSIFICATION MODELS
###########################################################
models = {
    "RandomForestClassifier": RandomForestClassifier(n_estimators=100, random_state=42),
    "GradientBoostingClassifier": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42),
    "SVC": SVC(C=1.0, kernel='rbf', probability=True),
    "XGBClassifier": XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    "LGBMClassifier": LGBMClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    "GaussianNB": GaussianNB(),  # <-- Added Bayesian Classification Model
    "BernoulliNB": BernoulliNB() # <-- Another Bayesian Model (good for binary data)
}

###########################################################
# 5) LOOP OVER FOLDS + SIMPLE BACKTEST (with LabelEncoder)
###########################################################
fold_results = {}

for fold_i, (X_train_fold, y_train_fold, X_test_fold, y_test_fold) in enumerate(folds, start=1):
    print(f"\n===== Fold {fold_i} =====")
    print(f"Train size: {len(X_train_fold)}, Test size: {len(X_test_fold)}")
    
    # 5.1 SHIFT labels with LabelEncoder so classes become consecutive integers
    le = LabelEncoder()
    
    # Fit on the training labels (which might be e.g. {-1,+1}, => SHIFT with le)
    # or {0,1} => SHIFT, etc. This ensures e.g. {0,2} becomes {0,1}.
    le.fit(y_train_fold)
    y_train_enc = le.transform(y_train_fold)
    
    # 5.2 Feature selection
    # We can pass the encoded y to select_features_rf_reg
    # because it just needs an estimator with .fit(X, y).
    # SHIFT isn't strictly necessary for random forest's .feature_importances_,
    # but let's be consistent.
    rf_for_fs = RandomForestClassifier(n_estimators=100, random_state=42)
    X_train_sel, selected_idx = select_features_rf_reg(
        X_train_fold, y_train_enc, estimator=rf_for_fs, max_features=20
    )
    feats = X_train_fold.columns[selected_idx]
    print(f"Selected features for Fold {fold_i}: {feats.tolist()}")

    X_test_sel = X_test_fold[feats]

    # 5.3 Scale
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_sel)
    X_test_scaled  = scaler.transform(X_test_sel)

    fold_results[fold_i] = {}

    for model_name, model in models.items():
        # 5.4 Train on the encoded training labels
        model.fit(X_train_scaled, y_train_enc)

        # 5.5 Predict encoded classes on test
        preds_enc = model.predict(X_test_scaled)
        
        # 5.6 Decode them back to the original label set (which might be e.g. [-1,0,+1])
        preds_fold = le.inverse_transform(preds_enc)

        # 5.7 Evaluate accuracy on the unencoded test labels
        acc = accuracy_score(y_test_fold, preds_fold)

        # 5.8 Convert classification => signals
        # If your original labels are: +1 => up, -1 => down, 0 => sideways,
        # then we interpret them as: +1 => buy, -1 => sell, 0 => no trade
        signals = preds_fold

        # Align with test portion
        df_test_fold = df_lbl.loc[X_test_fold.index].copy()

        # 5.9 Simple backtest with cost
        daily_returns, total_return = simulate_trading(signals, df_test_fold, cost=0.0002)
        sr = calculate_sharpe_ratio(np.array(daily_returns))

        fold_results[fold_i][model_name] = {
            "Accuracy": acc,
            "TotalReturn": total_return,
            "Sharpe": sr
        }

###########################################################
# 6) PRINT RESULTS
###########################################################
for fold_i, model_dict in fold_results.items():
    print(f"\n=== Fold {fold_i} Results ===")
    for model_name, stats in model_dict.items():
        acc = stats["Accuracy"]
        ret = stats["TotalReturn"]
        sr = stats["Sharpe"]
        print(f"{model_name}: ACC={acc:.2f}, Return={ret:.2f}%, Sharpe={sr:.2f}")


Number of folds created: 3

===== Fold 1 =====
Train size: 237, Test size: 237
Selected features for Fold 1: ['trend_kst_sig', 'trend_macd_signal', 'momentum_ppo_signal', 'trend_visual_ichimoku_b', 'momentum_tsi', 'trend_kst', 'trend_trix', 'trend_aroon_up', 'trend_visual_ichimoku_a', 'trend_macd', 'momentum_ppo', 'momentum_stoch_rsi_k', 'volatility_ui', 'trend_ichimoku_b', 'momentum_stoch_rsi_d', 'trend_ichimoku_conv', 'trend_mass_index', 'trend_adx_neg', 'momentum_wr', 'momentum_stoch_rsi']


  File "c:\Users\moham\miniconda3\envs\ml\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\moham\miniconda3\envs\ml\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\moham\miniconda3\envs\ml\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\moham\miniconda3\envs\ml\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


[LightGBM] [Info] Number of positive: 166, number of negative: 71
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000173 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1418
[LightGBM] [Info] Number of data points in the train set: 237, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.700422 -> initscore=0.849308
[LightGBM] [Info] Start training from score 0.849308
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

## Part 2: Model Training & Hyperparameter Tuning

**Objective:**  
Train an ML model (e.g., RandomForest, XGBoost) on the engineered features to predict the chosen labels.

**Tasks:**
- Perform time-based or walk-forward splits  
- Select top features if desired (e.g., using RandomForest feature importance)  
- Use `RandomizedSearchCV` or `GridSearchCV` to find optimal hyperparameters  
- Save the best model pipeline (e.g., `best_rf_pipeline.pkl`) 

In [ ]:
# Code 2: Hyperparameter Tuning for a Classification Model (Regime Detection)

import sys
import os
import warnings
from pathlib import Path

# ---------------------------------------------------------------------------
# 1) SET PROJECT ROOT AND UPDATE PATH/WORKING DIRECTORY
# ---------------------------------------------------------------------------
project_root = Path.cwd().parent.parent  # Adjust if your notebook is in notebooks/time_series
sys.path.append(str(project_root))
os.chdir(str(project_root))
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import MetaTrader5 as mt5
import joblib

# Sklearn / Models
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import accuracy_score, make_scorer
from sklearn.pipeline import Pipeline

# Your modules
from data.data_loader import get_data_mt5
from features.feature_engineering import add_all_ta_features
from features.labeling_schemes import create_labels_regime_detection  # or define inline




###########################################################
# 1) DATA LOADING & FEATURE ENGINEERING
###########################################################
if not mt5.initialize():
    print("Failed to initialize MT5")
else:
    data = get_data_mt5(symbol="BTCUSD", timeframe=mt5.TIMEFRAME_H4, n_bars=2000, start_pos=2000)
    mt5.shutdown()

df = add_all_ta_features(data)

###########################################################
# 2) Regime Detection Labeling
###########################################################

# Create classification labels for regime detection
df_lbl = create_labels_regime_detection(df, short_window=20, long_window=50)

# Now we have columns: 'ma_short', 'ma_long', 'regime_label' in {-1,0,+1}
X_full = df_lbl.drop(columns=["regime_label", "ma_short", "ma_long"])
y_full = df_lbl["regime_label"]

# SHIFT LABELS from [-1,0,+1] => [0,1,2]
# so the classifier won't complain about negative labels
y_full_shifted = y_full + 1  # -1->0, 0->1, +1->2

print("Unique classes in y_full:", y_full.unique())
print("Unique classes in y_full_shifted:", y_full_shifted.unique())

# Ensure chronological order if needed
# X_full = X_full.sort_index()
# y_full_shifted = y_full_shifted.loc[X_full.index]

###########################################################
# 3) DEFINE YOUR TRAIN PORTION
###########################################################
# e.g., first 80% for hyperparam tuning
split_idx = int(len(X_full) * 0.8)
X_tune = X_full.iloc[:split_idx].copy()
y_tune_shifted = y_full_shifted.iloc[:split_idx].copy()

print(f"Tuning portion size: {len(X_tune)}")

###########################################################
# 4) TIME-BASED CV (TimeSeriesSplit)
###########################################################
tscv = TimeSeriesSplit(n_splits=3)

# We'll define a scoring for classification (accuracy)
scorer = make_scorer(accuracy_score)

###########################################################
# 5) BUILD A PIPELINE
###########################################################
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(random_state=42))
])

###########################################################
# 6) DEFINE PARAM DISTRIBUTIONS FOR RandomForestClassifier
###########################################################
param_distributions = {
    "clf__n_estimators": [100, 200, 300],
    "clf__max_depth": [None, 5, 10, 15],
    "clf__min_samples_split": [2, 5, 10],
    "clf__max_features": ["auto", "sqrt", 0.5]
}

###########################################################
# 7) SET UP RandomizedSearchCV
###########################################################
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=10,               # how many random combos
    scoring=scorer,          # 'accuracy' metric
    cv=tscv,                 # time-based folds
    random_state=42,
    n_jobs=-1,
    verbose=2
)

###########################################################
# 8) FIT ON TUNING PORTION
###########################################################
random_search.fit(X_tune, y_tune_shifted)

print("Best params:", random_search.best_params_)
print("Best score (accuracy):", random_search.best_score_)

best_estimator = random_search.best_estimator_

###########################################################
# 9) SAVE THE BEST ESTIMATOR
###########################################################
joblib.dump(best_estimator, "models/saved_models/best_rf_rd_pipeline.pkl")
print("Saved best estimator to 'best_rf_rd_pipeline.pkl'")


Unique classes in y_full: [-1  1]
Unique classes in y_full_shifted: [0 2]
Tuning portion size: 1560
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params: {'clf__n_estimators': 200, 'clf__min_samples_split': 5, 'clf__max_features': 'sqrt', 'clf__max_depth': 10}
Best score (accuracy): 0.9290598290598292
Saved best estimator to 'best_rf_rd_pipeline.pkl'


## Part 3: Backtesting & Performance Evaluation

**Objective:**  
Evaluate how well the trained model performs on unseen data, simulating real trades.

**Tasks:**
- Use walk-forward or expanding splits to mimic “live” conditions  
- Convert model predictions to signals ([-1, 0, +1] or buy/sell/hold)  
- Run a simple backtest script or VectorBT for performance metrics  
- Calculate returns, Sharpe ratio, drawdowns, confusion matrix, etc.  
- Visualize results (equity curve, trades, etc.) to judge strategy viability  

In [ ]:
# THIRD CODE: Final Expanding Walk-Forward with VectorBT
# for a Regime Detection classification approach

import sys
import os
import warnings
from pathlib import Path

# ---------------------------------------------------------------------------
# 1) SET PROJECT ROOT AND UPDATE PATH/WORKING DIRECTORY
# ---------------------------------------------------------------------------
project_root = Path.cwd().parent.parent  # Adjust if your notebook is in notebooks/time_series
sys.path.append(str(project_root))
os.chdir(str(project_root))
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import MetaTrader5 as mt5
import vectorbt as vbt
import joblib

# Our modules
from data.data_loader import get_data_mt5
from features.feature_engineering import add_all_ta_features
from features.labeling_schemes import create_labels_regime_detection  # or define inline


# Sklearn
from sklearn.metrics import accuracy_score

###########################################################
# 1) DATA LOADING & FEATURE ENGINEERING
###########################################################
if not mt5.initialize():
    print("Failed to initialize MT5")
else:
    data = get_data_mt5(symbol="BTCUSD", timeframe=mt5.TIMEFRAME_H4, n_bars=2000, start_pos=0)
    mt5.shutdown()

# 1.1 Add technical features
df = add_all_ta_features(data)

# 1.2 Create regime detection labels
df_lbl = create_labels_regime_detection(df, short_window=20, long_window=50)

# We'll treat 'regime_label' in {-1,0,+1} as a 3-class classification
X = df_lbl.drop(columns=["regime_label", "ma_short", "ma_long"])
y = df_lbl["regime_label"]

# SHIFT from [-1,0,+1] => [0,1,2] for scikit-learn classification
y_shifted = y + 1  # -1->0, 0->1, +1->2

###########################################################
# 2) DEFINING EXPANDING SPLITS
###########################################################
def expanding_walk_forward_splits(X, y, n_splits=3):
    """
    Creates expanding walk-forward folds. For each fold i:
      - Train: [0 : fold_i]
      - Test:  [fold_i : fold_{i+1}]
    The last fold extends to the end.
    """
    n = len(X)
    fold_size = n // (n_splits + 1)
    splits = []
    
    for i in range(n_splits):
        train_end = (i+1) * fold_size
        if i == n_splits - 1:
            test_end = n
        else:
            test_end = (i+2) * fold_size
            if test_end > n:
                test_end = n
        
        X_train_fold = X.iloc[:train_end]
        y_train_fold = y.iloc[:train_end]
        
        X_test_fold = X.iloc[train_end:test_end]
        y_test_fold = y.iloc[train_end:test_end]
        
        splits.append((X_train_fold, y_train_fold, X_test_fold, y_test_fold))
    return splits

folds = expanding_walk_forward_splits(X, y_shifted, n_splits=3)
print(f"Number of folds: {len(folds)}")

###########################################################
# 3) LOAD BEST PIPELINE & RUN EXPANDING WALK-FORWARD
###########################################################
# This pipeline was presumably tuned on SHIFTED classes [0,1,2].
best_pipeline = joblib.load("models/saved_models/best_rf_rd_pipeline.pkl")
print("Loaded best pipeline from 'best_rf_rd_pipeline.pkl'")

fees = 0.0002  # 0.02% transaction cost per trade

fold_results = {}

for i, (X_train_fold, y_train_fold, X_test_fold, y_test_fold) in enumerate(folds, start=1):
    print(f"\n=== Fold {i} ===")
    print(f"Train size: {len(X_train_fold)}, Test size: {len(X_test_fold)}")
    
    # 3.1 Fit the pipeline on SHIFTED labels in this fold's train portion
    best_pipeline.fit(X_train_fold, y_train_fold)
    
    # 3.2 Predict SHIFTED labels on the test portion
    preds_shifted = best_pipeline.predict(X_test_fold)
    
    # 3.3 SHIFT back to [-1,0,+1]
    preds = preds_shifted - 1
    
    # Evaluate accuracy vs. unshifted test labels
    # The test labels are SHIFTED, so let's SHIFT them back
    y_test_fold_unshifted = y_test_fold - 1
    acc = accuracy_score(y_test_fold_unshifted, preds)
    print(f"Fold {i} Accuracy={acc:.2f}")
    
    # Convert predicted classes => signals
    # +1 => buy, -1 => sell, 0 => no position
    signals = preds
    
    # Vectorbt backtest on the test portion
    df_test_fold = df_lbl.loc[X_test_fold.index].copy()
    close_prices = df_test_fold["close"]
    
    if len(signals) < len(close_prices):
        # pad signals if needed
        signals = np.append(signals, [0]*(len(close_prices)-len(signals)))
    signals_s = pd.Series(signals, index=close_prices.index)
    
    pf = vbt.Portfolio.from_signals(
        close_prices,
        entries=signals_s > 0,
        exits=signals_s < 0,
        init_cash=10000,
        freq='4H',
        fees=fees
    )
    
    total_return = pf.total_return()
    sharpe_ratio = pf.sharpe_ratio()
    
    print(f"Fold {i} Return={total_return:.2f}%, Sharpe={sharpe_ratio:.2f}")
    print("\nVectorbt stats for Fold", i)
    print(pf.stats())
    
    # Optional: Plot
    fig = pf.plot()
    fig.show()
    
    # Store results
    fold_results[i] = {
        "Accuracy": acc,
        "TotalReturn": total_return,
        "Sharpe": sharpe_ratio
    }

###########################################################
# 4) PRINT FOLD RESULTS
###########################################################
for i, stats in fold_results.items():
    acc = stats["Accuracy"]
    ret = stats["TotalReturn"]
    sr = stats["Sharpe"]
    print(f"\nFold {i} => ACC={acc:.2f}, Return={ret:.2f}%, Sharpe={sr:.2f}")


Number of folds: 3
Loaded best pipeline from 'best_rf_rd_pipeline.pkl'

=== Fold 1 ===
Train size: 487, Test size: 487
Fold 1 Accuracy=0.81
Fold 1 Return=-0.14%, Sharpe=-2.17

Vectorbt stats for Fold 1
Start                         2024-06-23 12:00:00
End                           2024-09-12 12:00:00
Period                           81 days 04:00:00
Start Value                               10000.0
End Value                             8634.473296
Total Return [%]                       -13.655267
Benchmark Return [%]                   -10.340619
Max Gross Exposure [%]                      100.0
Total Fees Paid                         22.357979
Max Drawdown [%]                        15.791025
Max Drawdown Duration            52 days 12:00:00
Total Trades                                    6
Total Closed Trades                             6
Total Open Trades                               0
Open Trade PnL                                0.0
Win Rate [%]                            33.33333


=== Fold 2 ===
Train size: 974, Test size: 487
Fold 2 Accuracy=0.97
Fold 2 Return=0.24%, Sharpe=2.66

Vectorbt stats for Fold 2
Start                         2024-09-12 16:00:00
End                           2024-12-02 16:00:00
Period                           81 days 04:00:00
Start Value                               10000.0
End Value                            12384.606547
Total Return [%]                        23.846065
Benchmark Return [%]                    62.919084
Max Gross Exposure [%]                      100.0
Total Fees Paid                         23.447974
Max Drawdown [%]                        13.868133
Max Drawdown Duration            44 days 20:00:00
Total Trades                                    6
Total Closed Trades                             5
Total Open Trades                               1
Open Trade PnL                        -249.260814
Win Rate [%]                                 60.0
Best Trade [%]                          23.906063
Worst Trade [%]      


=== Fold 3 ===
Train size: 1461, Test size: 490
Fold 3 Accuracy=0.93
Fold 3 Return=-0.18%, Sharpe=-2.05

Vectorbt stats for Fold 3
Start                         2024-12-02 20:00:00
End                           2025-02-22 08:00:00
Period                           81 days 16:00:00
Start Value                               10000.0
End Value                             8244.771537
Total Return [%]                       -17.552285
Benchmark Return [%]                     1.011729
Max Gross Exposure [%]                      100.0
Total Fees Paid                         34.097529
Max Drawdown [%]                        21.074684
Max Drawdown Duration            66 days 16:00:00
Total Trades                                   10
Total Closed Trades                             9
Total Open Trades                               1
Open Trade PnL                         -46.901194
Win Rate [%]                            33.333333
Best Trade [%]                           2.523353
Worst Trade [%]   


Fold 1 => ACC=0.81, Return=-0.14%, Sharpe=-2.17

Fold 2 => ACC=0.97, Return=0.24%, Sharpe=2.66

Fold 3 => ACC=0.93, Return=-0.18%, Sharpe=-2.05


Backtest with full dataset

In [1]:
import sys
import os
import warnings
from pathlib import Path

# ---------------------------------------------------------------------------
# 1) SET PROJECT ROOT AND UPDATE PATH/WORKING DIRECTORY
# ---------------------------------------------------------------------------
project_root = Path.cwd().parent.parent
sys.path.append(str(project_root))
os.chdir(str(project_root))
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import MetaTrader5 as mt5
import vectorbt as vbt
import joblib

# Our modules
from data.data_loader import get_data_mt5
from features.feature_engineering import add_all_ta_features
from features.labeling_schemes import create_labels_regime_detection  # Regime detection function

# Sklearn
from sklearn.metrics import accuracy_score

###########################################################
# 1) DATA LOADING & FEATURE ENGINEERING
###########################################################
if not mt5.initialize():
    print("Failed to initialize MT5")
else:
    # Fetch 2000 most recent bars for backtesting
    data = get_data_mt5(symbol="BTCUSD", timeframe=mt5.TIMEFRAME_H4, n_bars=2000, start_pos=0)
    mt5.shutdown()

# Add technical features
df = add_all_ta_features(data)

# Create regime detection labels
df_lbl = create_labels_regime_detection(df, short_window=20, long_window=50)

# X => all features except the label column
X = df_lbl.drop(columns=["regime_label", "ma_short", "ma_long"])  # Remove moving averages if used
y = df_lbl["regime_label"]

# SHIFT labels from [-1,0,+1] => [0,1,2] to avoid Sklearn classification errors
y_shifted = y + 1  # -1 → 0, 0 → 1, +1 → 2

###########################################################
# 2) LOAD BEST CLASSIFICATION MODEL & TRAIN ON FULL DATASET
###########################################################
best_pipeline = joblib.load("models/saved_models/best_rf_rd_pipeline.pkl")
print("Loaded best classification model from 'best_rf_rd_pipeline.pkl'")

print("\nTraining on full dataset (No folds) ...")
best_pipeline.fit(X, y_shifted)  # Train on full dataset

# Predict on full dataset
preds_shifted = best_pipeline.predict(X)

# Shift predictions back to [-1,0,+1] class labels
preds = preds_shifted - 1

# Compute accuracy score against true labels (after shifting back)
y_true_unshifted = y_shifted - 1
accuracy = accuracy_score(y_true_unshifted, preds)
print(f"\nFull Dataset Accuracy: {accuracy:.4f}")

# Convert predicted classes => signals
# +1 => buy, -1 => sell, 0 => no position
signals = preds

###########################################################
# 3) RUN VECTORBT BACKTEST ON FULL DATASET
###########################################################
print("\nRunning Full Backtest on Entire 2000 Bars...")

# Align signals with the dataset
df_full = df_lbl.copy()
close_prices_full = df_full["close"]

if len(signals) < len(close_prices_full):
    signals = np.append(signals, [0] * (len(close_prices_full) - len(signals)))

signals_s = pd.Series(signals, index=close_prices_full.index)

fees = 0.0002  # 0.02% transaction cost per trade

pf_full = vbt.Portfolio.from_signals(
    close_prices_full,
    entries=signals_s > 0,
    exits=signals_s < 0,
    init_cash=10000,
    freq='4H',
    fees=fees
)

total_return = pf_full.total_return()
sharpe_ratio = pf_full.sharpe_ratio()

print("\nFull Backtest Results:")
print(f"Accuracy={accuracy:.2f}, Return={total_return:.2f}%, Sharpe={sharpe_ratio:.2f}")
print(pf_full.stats())

# Optional: Plot the backtest results
fig = pf_full.plot()
fig.show()


Loaded best classification model from 'best_rf_rd_pipeline.pkl'

Training on full dataset (No folds) ...

Full Dataset Accuracy: 1.0000

Running Full Backtest on Entire 2000 Bars...

Full Backtest Results:
Accuracy=1.00, Return=-0.10%, Sharpe=-0.17
Start                               2024-04-10 16:00:00
End                                 2025-03-01 16:00:00
Period                                325 days 04:00:00
Start Value                                     10000.0
End Value                                   8969.677687
Total Return [%]                             -10.303223
Benchmark Return [%]                          22.493886
Max Gross Exposure [%]                            100.0
Total Fees Paid                               98.555269
Max Drawdown [%]                              24.262087
Max Drawdown Duration                 213 days 12:00:00
Total Trades                                         26
Total Closed Trades                                  26
Total Open Trades      